# Klasik Optimizasyon Problemleri - MealPy Uygulamaları

Bu derste amaç, daha önce kavramsal olarak incelediğimiz üç klasik problemi Mealpy kullanarak çözmektir:


- Gezgin Satıcı Problemi (TSP),

- Sırt Çantası Problemi (Knapsack),
- Çizelgeleme Problemi

Bu uygulamalarda problemleri kodla tanımlayacağız Mealpy ile farklı algoritmalar deneyeceğiz, en iyi çözümü yorumlayacağız, yakınsama eğrilerini karşılaştıracağız.

Kütüphaneye [Bu Linkten](https://github.com/thieu1995/mealpy/tree/master) ulaşabilirsiniz.

# Install Colab

In [ ]:
!pip install mealpy

# Import Library

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mealpy import FloatVar, BinaryVar, PermutationVar
from mealpy import GA, GWO, PSO

# Necessary Function

In [ ]:
def plot_histories(histories, title="Yakınsama Eğrisi"):
    plt.figure(figsize=(8, 5))
    for name, hist in histories.items():
        plt.plot(hist, label=name)
    plt.xlabel("İterasyon")
    plt.ylabel("En iyi fitness")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


def run_optimizers(problem_dict, optimizers):
    results = []
    histories = {}
    solutions = {}

    for name, model in optimizers.items():
        g_best = model.solve(problem_dict)
        best_fit = g_best.target.fitness
        best_sol = g_best.solution

        results.append({
            "Algoritma": name,
            "En İyi Fitness": best_fit,
            "En İyi Çözüm": best_sol
        })

        histories[name] = model.history.list_global_best_fit
        solutions[name] = {
            "best_fitness": best_fit,
            "best_solution": best_sol
        }

        print(f"\n{name}")
        print("-" * len(name))
        print("Best fitness:", best_fit)
        print("Best solution:", best_sol)

    df = pd.DataFrame(results).sort_values("En İyi Fitness").reset_index(drop=True)
    return df, histories, solutions

# Problem 1: Gezgin Satıcı Problemi (TSP)

## Problem tanımı

Bir teknik servis ekibi 12 farklı hastaneyi dolaşacak ve başladığı merkeze geri dönecektir. Amaç, toplam rota uzunluğunu minimize etmektir.

- **Karar değişkeni:** şehir sırası, yani bir permütasyon

- **Amaç fonksiyonu:** toplam turun uzunluğu

## Veri oluşturma

In [ ]:
np.random.seed(42)

n_cities = 12
city_names = [f"H{i}" for i in range(n_cities)]

# 2B koordinatlar
coords = np.random.uniform(0, 100, size=(n_cities, 2))

# Mesafe matrisi
dist_matrix = np.zeros((n_cities, n_cities))
for i in range(n_cities):
    for j in range(n_cities):
        dist_matrix[i, j] = np.linalg.norm(coords[i] - coords[j])

coord_df = pd.DataFrame(coords, columns=["x", "y"], index=city_names)
coord_df

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(coords[:, 0], coords[:, 1], s=80)

for i, name in enumerate(city_names):
    plt.text(coords[i, 0] + 1, coords[i, 1] + 1, name)

plt.title("TSP Şehir Konumları")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()

## Amaç fonksiyonu

In [ ]:
def tsp_objective(solution):
    route = np.array(solution, dtype=int)
    total_distance = 0.0

    for i in range(len(route) - 1):
        total_distance += dist_matrix[route[i], route[i + 1]]

    # Başlangıca dönüş
    total_distance += dist_matrix[route[-1], route[0]]
    return total_distance

## Mealpy problem tanımı

Permutation tabanlı problem olduğu için **PermutationVar** kullanıyoruz. Mealpy’nin güncel dokümantasyonunda permütasyon tabanlı kombinatoryal problemler için bu değişken sınıfı öneriliyor.

In [ ]:
tsp_problem = {
    "bounds": PermutationVar(valid_set=tuple(range(n_cities)), name="route"),
    "obj_func": tsp_objective,
    "minmax": "min",
    "save_population": False,
}

## Farklı algoritmalarla çözüm

In [ ]:
tsp_optimizers = {
    "GA": GA.BaseGA(epoch=300, pop_size=60),
    "PSO": PSO.OriginalPSO(epoch=300, pop_size=60),
    "GWO": GWO.OriginalGWO(epoch=300, pop_size=60)
}

In [ ]:
tsp_results, tsp_histories, tsp_solutions = run_optimizers(tsp_problem, tsp_optimizers)

In [ ]:
tsp_results

In [ ]:
plot_histories(tsp_histories, title="TSP - Yakınsama Eğrileri")

## En iyi rotayı çizdirme

In [ ]:
best_model_name = tsp_results.iloc[0]["Algoritma"]
best_model = tsp_optimizers[best_model_name]
best_route = np.array(best_model.g_best.solution, dtype=int)

print("En iyi algoritma:", best_model_name)
print("En iyi rota indeksleri:", best_route)
print("En iyi rota isimleri:", [city_names[i] for i in best_route])
print("Toplam mesafe:", best_model.g_best.target.fitness)

In [ ]:
route_coords = coords[best_route]
route_coords_closed = np.vstack([route_coords, route_coords[0]])

plt.figure(figsize=(7, 7))
plt.scatter(coords[:, 0], coords[:, 1], s=80)

for i, name in enumerate(city_names):
    plt.text(coords[i, 0] + 1, coords[i, 1] + 1, name)

plt.plot(route_coords_closed[:, 0], route_coords_closed[:, 1], marker="o")
plt.title(f"TSP En İyi Rota ({best_model_name})")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(alpha=0.3)
plt.show()

# Problem 2: Sırt Çantası Problemi (Knapsack)

## Problem tanımı

Bir acil sağlık drone’u 20 farklı tıbbi malzeme arasından seçim yapacaktır.

Her malzemenin ağırlığı ve önemi/değeri vardır.

- **Amaç:** kapasiteyi aşmadan toplam değeri maksimize etmek
- **Karar değişkeni:** her malzeme seçildi mi seçilmedi mi, yani binary yapı

## Veri oluşturma

In [ ]:
np.random.seed(123)

n_items = 20
weights = np.random.randint(2, 15, size=n_items)
values = np.random.randint(10, 100, size=n_items)

capacity = int(weights.sum() * 0.35)

items_df = pd.DataFrame({
    "Malzeme": [f"M{i+1}" for i in range(n_items)],
    "Ağırlık": weights,
    "Değer": values
})

print("Kapasite:", capacity)
items_df

## Amaç fonksiyonu

In [ ]:
def knapsack_objective(solution):
    x = np.array(solution, dtype=int)

    total_weight = np.sum(weights * x)
    total_value = np.sum(values * x)

    penalty = 0.0
    if total_weight > capacity:
        penalty = 1000 * (total_weight - capacity)

    # Maksimizasyonu minimizasyona çevirmek için eksi
    return -total_value + penalty

## Mealpy Problem tanımı

In [ ]:
knapsack_problem = {
    "bounds": BinaryVar(n_vars=n_items, name="pick"),
    "obj_func": knapsack_objective,
    "minmax": "min",
    "log_to": None,
    "save_population": False,
}

## Farklı algoritmalarla çözüm

In [ ]:
knapsack_optimizers = {
    "GA": GA.BaseGA(epoch=300, pop_size=60),
    "PSO": PSO.OriginalPSO(epoch=300, pop_size=60),
    "GWO": GWO.OriginalGWO(epoch=300, pop_size=60)
}

In [ ]:
knapsack_results, knapsack_histories, knapsack_solutions = run_optimizers(knapsack_problem, knapsack_optimizers)

In [ ]:
knapsack_results

In [ ]:
plot_histories(knapsack_histories, title="Knapsack - Yakınsama Eğrileri")

Knapsack çözümünün mantıksal temsili 0 ve 1’lerden oluşmalı!

* GA → binary probleme doğal daha yakın

* PSO / GWO → continuous algoritmalar

* Knapsack gibi ayrık problemlerde ek bir eşikleme / yuvarlama / binary mapping gerekir.


PSO ve GWO çıktılarının continuous gelmesinin nedeni, bu algoritmaların özünde sürekli uzayda çalışan optimizasyon yöntemleri olmasıdır.

Bu tarz durumlarda çözümü yorumlamadan önce binary’ye dönüşüm işlemi gerçekleştiririz.



```
best_x_raw = best_model.g_best.solution
best_x = (np.array(best_x_raw) > 0.5).astype(int)
```



## En iyi çözümü yorumlama

In [ ]:
best_model_name = knapsack_results.iloc[0]["Algoritma"]
best_model = knapsack_optimizers[best_model_name]

best_x_raw = np.array(best_model.g_best.solution)

# Continuous çıktıyı binary'ye dönüştür
best_x = np.floor(best_x_raw).astype(int)
best_x = np.clip(best_x, 0, 1)

selected_items = items_df.copy()
selected_items["Seçildi"] = best_x
selected_items = selected_items[selected_items["Seçildi"] == 1].reset_index(drop=True)

total_weight = np.sum(weights * best_x)
total_value = np.sum(values * best_x)

print("En iyi algoritma:", best_model_name)
print("Ham çözüm:", best_x_raw)
print("Binary çözüm:", best_x)
print("Toplam ağırlık:", total_weight)
print("Toplam değer:", total_value)
print("Kapasite:", capacity)

selected_items

# Problem 3: Çizelgeleme

## Problem tanımı

Bir hastanede 10 ameliyat/iş tek ameliyathanede yapılacaktır. Her işin süresi farklıdır.

**Amaç:** toplam tamamlanma sürelerini minimize etmek

Bu, önceki haftalarda anlattığınız kümülatif toplam mantığının daha büyük boyutlu bir versiyonudur.

**Karar değişkeni:** iş sırası, yani permütasyon

## Veri oluşturma

In [ ]:
np.random.seed(7)

n_jobs = 10
job_names = [f"J{i+1}" for i in range(n_jobs)]
processing_times = np.random.randint(1, 12, size=n_jobs)

jobs_df = pd.DataFrame({
    "İş": job_names,
    "Süre": processing_times
})

jobs_df

## Amaç fonksiyonu

In [ ]:
def scheduling_objective(solution):
    order = np.array(solution, dtype=int)

    cumulative_time = 0
    total_completion_time = 0

    for job_idx in order:
        cumulative_time += processing_times[job_idx]
        total_completion_time += cumulative_time

    return total_completion_time

## Melapy Problem tanımı

In [ ]:
scheduling_problem = {
    "bounds": PermutationVar(valid_set=tuple(range(n_jobs)), name="order"),
    "obj_func": scheduling_objective,
    "minmax": "min",
    "save_population": False,
}

## Farklı algoritmalarla çözüm

In [ ]:
scheduling_optimizers = {
    "GA": GA.BaseGA(epoch=300, pop_size=60),
    "PSO": PSO.OriginalPSO(epoch=300, pop_size=60),
    "GWO": GWO.OriginalGWO(epoch=300, pop_size=60)
}

In [ ]:
scheduling_results, scheduling_histories, scheduling_solutions = run_optimizers(scheduling_problem, scheduling_optimizers)
scheduling_results

In [ ]:
scheduling_results

In [ ]:
plot_histories(scheduling_histories, title="Çizelgeleme - Yakınsama Eğrileri")

## En iyi sırayı yorumlama

In [ ]:
best_model_name = scheduling_results.iloc[0]["Algoritma"]
best_model = scheduling_optimizers[best_model_name]
best_order = np.array(best_model.g_best.solution, dtype=int)

ordered_jobs = jobs_df.iloc[best_order].copy().reset_index(drop=True)

completion_times = []
cum = 0
for t in ordered_jobs["Süre"]:
    cum += t
    completion_times.append(cum)

ordered_jobs["Tamamlanma Süresi"] = completion_times

print("En iyi algoritma:", best_model_name)
print("Toplam tamamlanma süresi:", best_model.g_best.target.fitness)

ordered_jobs

# Problem 4: Araç Tasarımında Yakıt Tüketimi Optimizasyonu

Bir aracın yakıt tüketimini minimize etmek istiyoruz. Yakıt tüketimi birçok fiziksel ve tasarım parametresine bağlıdır.

## Problem Tanımı

Bir otomobil üreticisi, bir aracın yakıt tüketimini minimize etmek istiyor.

Kontrol edilebilir parametreler:

| Değişken | Açıklama                        |
| -------- | ------------------------------- |
| ( x_1 )  | Araç ağırlığı (kg)              |
| ( x_2 )  | Motor hacmi (L)                 |
| ( x_3 )  | Aerodinamik sürükleme katsayısı |
| ( x_4 )  | Lastik basıncı (psi)            |
| ( x_5 )  | Vites oranı                     |
| ( x_6 )  | Motor verimliliği (%)           |
| ( x_7 )  | Araç hızı (km/h)                |




Karar değişkeni sınırları:

`800 ≤ x1 ≤ 2000`

`1.0 ≤ x2 ≤ 3.5`

`0.20 ≤ x3 ≤ 0.50`

`28 ≤ x4 ≤ 40`

`2.5 ≤ x5 ≤ 5.0`

`60 ≤ x6 ≤ 95`

`40 ≤ x7 ≤ 140`


* Amaç:

👉 Yakıt tüketimini minimize etmek

## Veri oluşturma

Her problemde veri olmaz.


## Amaç fonksiyonu


Aşağıdaki basitleştirilmiş model kullanılır:



```
f(x) = 0.0005*x1 + 2*x2 + 15*x3 + 0.01*(x7 - 80)**2 - 0.05*x6 + 0.1*x5 + 0.02*(35 - x4)**2
```

Amaç:

` min f(x) `

Burada:

* araç ağırlığı arttıkça tüketim artar,

* motor hacmi arttıkça tüketim artar,

* aerodinamik sürükleme arttıkça tüketim artar,

* hızın 80 km/h civarında olması avantaj sağlar,

* motor verimliliği arttıkça tüketim azalır,

* lastik basıncının 35 psi civarında olması avantaj sağlar,



In [ ]:
def fuel_consumption_objective(solution):
    x1, x2, x3, x4, x5, x6, x7 = solution

    fuel = (
        0.0005 * x1 +
        2.0 * x2 +
        15.0 * x3 +
        0.01 * (x7 - 80)**2 -
        0.05 * x6 +
        0.1 * x5 +
        0.02 * (35 - x4)**2
    )

    return fuel

## Melpy Problem tanımı

In [ ]:
problem_dict = {
    "bounds": FloatVar(
        lb=[800, 1.0, 0.20, 28, 2.5, 60, 40],
        ub=[2000, 3.5, 0.50, 40, 5.0, 95, 140],
        name="fuel_params"
    ),
    "obj_func": fuel_consumption_objective,
    "minmax": "min"
}

## Farklı algoritmalar ile çözüm

In [ ]:
optimizers = {
    "GA": GA.BaseGA(epoch=200, pop_size=50),
    "PSO": PSO.OriginalPSO(epoch=200, pop_size=50),
    "GWO": GWO.OriginalGWO(epoch=200, pop_size=50)
}

In [ ]:
engineering_results, engineering_histories, engineering_solutions = run_optimizers(problem_dict,
                                                                                   optimizers)

In [ ]:
engineering_results

## En iyi çözümü yorumlama

In [ ]:
plot_histories(engineering_histories, title="Yakıt Tüketimi Problemi için Yakınsama Eğrisi")

In [ ]:
best_algorithm_name = engineering_results.iloc[0]["Algoritma"]
best_model = optimizers[best_algorithm_name]
best_solution = best_model.g_best.solution
best_fitness = best_model.g_best.target.fitness

print("En iyi algoritma:", best_algorithm_name)
print("En iyi fitness:", best_fitness)
print("En iyi çözüm:", best_solution)

In [ ]:
x1, x2, x3, x4, x5, x6, x7 = best_solution

print(f"Araç ağırlığı (kg): {x1:.2f}")
print(f"Motor hacmi (L): {x2:.2f}")
print(f"Aerodinamik sürükleme katsayısı: {x3:.4f}")
print(f"Lastik basıncı (psi): {x4:.2f}")
print(f"Vites oranı: {x5:.2f}")
print(f"Motor verimliliği (%): {x6:.2f}")
print(f"Araç hızı (km/h): {x7:.2f}")

# ÖZET

Bu uygulamalarda gördüğümüz gibi:

Problem tanımı değiştikçe karar değişkeni tipi de değişmektedir.

- TSP ve çizelgeleme permütasyon tabanlıdır.
- Knapsack binary seçim problemidir.
- Mühendislik problemleri (genellikle) sürekli problemlerdir.

Mealpy, uygun değişken türü tanımlandığında bu farklı problem yapılarını aynı çerçevede çözmeye imkân verir.

Algoritmaların ortak mantığı: aday çözüm üretmek, fitness hesaplamak ve çözümleri iteratif olarak iyileştirmektir. history nesnesi üzerinden yakınsama ve performans takibi yapılabilir

# ÖDEV

- Ödev 1'de Biyomedikal alanda tasarladığınız klasik optimizasyon problemini problemin boyutunu (En az 10 boyutlu olacak şekilde) genişletiniz. Colab üzerinde problemi, karar değişkenlerini, amaç fonksiyonunu tanımlayınız.

- Veriler oluşturulurken random değerler yerine gerçek veriler kullanınız. Örneğin TSP için Google Maps'ten koordinatlar çekilebilir. Cihaz satın alma, ameliyathane vs için Google üzerinden küçük bir araştırma yapınız.

- Tasarladığınız problemi yukarıdaki optimizasyon algoritmaları (Ödev 2'de sorumlu olduğunuz **Optimizasyon Algoritmasını** da ekleyiniz.) ile çözünüz. Sonuçları karşılaştırınız.


**Not:** Öğrenciler aşağıdaki örnek tablo içindeki satırları artırabilir, değerleri değiştirebilir ve kendi problemlerini oluşturabilir. Kodun geri kalanı aynı mantıkla çalışacaktır.

## TSP örnek dataframe

In [ ]:
import pandas as pd

tsp_df = pd.DataFrame({
    "Konum": [
        "Hastane A",
        "Hastane B",
        "Hastane C",
        "Hastane D",
        "Hastane E",
        "Hastane F",
        "Hastane G",
        "Hastane H",
        "Hastane I",
        "Hastane J"
    ],
    "Enlem": [
        36.5872,
        36.5841,
        36.5798,
        36.5925,
        36.6013,
        36.5954,
        36.6062,
        36.6118,
        36.5764,
        36.5897
    ],
    "Boylam": [
        36.1735,
        36.1621,
        36.1814,
        36.1553,
        36.1688,
        36.1912,
        36.1777,
        36.1605,
        36.1709,
        36.1846
    ]
})

tsp_df

In [ ]:
coords = tsp_df[["Enlem", "Boylam"]].values
coords

## Knapsack örnek dataframe

In [ ]:
knapsack_df = pd.DataFrame({
    "Urun": [
        "Cihaz 1",
        "Cihaz 2",
        "Cihaz 3",
        "Cihaz 4",
        "Cihaz 5",
        "Cihaz 6",
        "Cihaz 7",
        "Cihaz 8",
        "Cihaz 9",
        "Cihaz 10"
    ],
    "Maliyet": [
        85000,
        120000,
        45000,
        70000,
        25000,
        150000,
        95000,
        30000,
        18000,
        5000
    ],
    "FaydaPuani": [
        80,
        95,
        70,
        85,
        60,
        98,
        75,
        55,
        50,
        40
    ]
})

knapsack_df

In [ ]:
# bütçe
budget = 300000

values = knapsack_df["FaydaPuani"].values
weights = knapsack_df["Maliyet"].values

## Çizelgeleme için örnek dataframe

In [ ]:
schedule_df = pd.DataFrame({
    "Is": [
        "Ameliyat 1",
        "Ameliyat 2",
        "Ameliyat 3",
        "Ameliyat 4",
        "Ameliyat 5",
        "Ameliyat 6",
        "Ameliyat 7",
        "Ameliyat 8",
        "Ameliyat 9",
        "Ameliyat 10"
    ],
    "IslemSuresi_dk": [
        45,
        120,
        60,
        90,
        30,
        150,
        75,
        50,
        110,
        40
    ],
    "Oncelik": [
        2,
        1,
        3,
        1,
        4,
        1,
        2,
        3,
        2,
        4
    ]
})

schedule_df

In [ ]:
processing_times = schedule_df["IslemSuresi_dk"].values
processing_times